In [1]:
import torch
import model
import lancedb
import os
from hydra import initialize, initialize_config_module, initialize_config_dir, compose
from omegaconf import OmegaConf
from dotenv import load_dotenv
from datafusion import functions as f
from torch.utils.data import DataLoader, random_split
import torch.nn.functional as F
import numpy as np
from PIL import Image
from torchvision import transforms
import multiprocessing
import trackio
from tqdm.auto import tqdm

In [2]:
with initialize(version_base=None, config_path="conf"):
    load_dotenv()
    write_url = os.environ["TRACKIO_WRITE_URL"]
    cfg = compose(config_name="config", overrides=[f'trackio.write_url="{write_url}"'])
    trackio.init(name=cfg.trackio.run_name, project=cfg.trackio.project, server_url=cfg.trackio.write_url)

Task was destroyed but it is pending!
task: <Task pending name='Task-38' coro=<_async_in_context.<locals>.run_in_context() running at /Users/matheoledevehat/Code/worldmodels/ha_schmidhuber/.venv/lib/python3.11/site-packages/ipykernel/utils.py:57> cb=[ZMQStream._run_callback.<locals>._log_error() at /Users/matheoledevehat/Code/worldmodels/ha_schmidhuber/.venv/lib/python3.11/site-packages/zmq/eventloop/zmqstream.py:563]>
/Users/matheoledevehat/.local/share/uv/python/cpython-3.11.11-macos-aarch64-none/lib/python3.11/asyncio/base_events.py:679: RuntimeWarning: coroutine '_async_in_context.<locals>.run_in_context' was never awaited
  self._ready.clear()


* Trackio project initialized: ha_schmidhuber
* Trackio metrics will be sent to self-hosted server: https://trackio.flydexo.com


* Apple Silicon detected, enabling automatic GPU/system metrics logging
* psutil detected, enabling automatic CPU/system metrics logging
* Created new run: vae{beta=sigmoid,epochs=1}


In [3]:
vae = model.AutoEncoder(cfg).to(cfg.device)

In [ ]:
class ShuffledFrameDataset(torch.utils.data.IterableDataset):
    """Streams episodes from LanceDB and yields single frames in shuffled order.

    Frames within an episode are highly correlated, so batching whole episodes gives
    poor gradients. Loading every frame in RAM is impossible (10k × 1000 frames ≈ 123 GB)
    and reading one ~12 MB episode row per frame would be a 1000× read amplification.
    A shuffle buffer gets both: each episode is read once per pass, its frames are mixed
    into a pool of `buffer_frames` frames spanning ~dozens of episodes, and frames are
    drawn from the pool at random (~250 MB RAM at the default size).
    """

    def __init__(self, cfg, episode_ids, buffer_frames=20_000, seed=0):
        super().__init__()
        self.lancedb_uri = cfg.dataset.lancedb_uri
        self.img_size = cfg.dataset.img_size
        self.episode_ids = np.asarray(episode_ids)
        self.buffer_frames = buffer_frames
        self.seed = seed
        # Estimate: episodes rarely terminate before the step limit, so this is exact
        # for almost every episode. Only used for len(loader) / progress / schedules.
        self.est_frames = len(self.episode_ids) * cfg.dataset.max_episode_steps

    def __len__(self):
        return self.est_frames

    def _episode_frames(self, table, ep_id):
        row = table.search().where(f'episode_id = {ep_id}').limit(1).to_arrow()
        frames = np.array(row.column('observations').combine_chunks()[0].as_py(), dtype=np.uint8)
        return frames.reshape(-1, self.img_size, self.img_size, 3)

    def __iter__(self):
        # _pass survives across epochs inside persistent workers → fresh order every epoch
        self._pass = getattr(self, "_pass", -1) + 1
        info = torch.utils.data.get_worker_info()
        wid, nw = (info.id, info.num_workers) if info else (0, 1)
        rng = np.random.default_rng((self.seed, self._pass, wid))
        ids = self.episode_ids[wid::nw].copy()   # disjoint episodes per worker
        rng.shuffle(ids)
        table = lancedb.connect(self.lancedb_uri).open_table("episodes")

        buffer, it, exhausted = [], iter(ids), False
        while True:
            while not exhausted and len(buffer) < self.buffer_frames:
                ep = next(it, None)
                if ep is None:
                    exhausted = True
                else:
                    buffer.extend(self._episode_frames(table, int(ep)))
            if not buffer:
                return
            j = rng.integers(len(buffer))
            buffer[j], buffer[-1] = buffer[-1], buffer[j]   # swap-pop a random frame
            frame = buffer.pop()
            # copy() detaches the 12 KB frame from its 12 MB parent episode array
            yield torch.from_numpy(frame.copy()).permute(2, 0, 1).float() / 255.0

In [ ]:
# Split by *episode*, not by frame — neighbouring frames are near-duplicates, so a
# frame-level split would leak train data into val.
n_episodes = lancedb.connect(cfg.dataset.lancedb_uri).open_table("episodes").count_rows()
perm = np.random.default_rng(0).permutation(n_episodes)
n_val = max(1, int(0.1 * n_episodes))
val_ids, train_ids = perm[:n_val], perm[n_val:]

train_ds = ShuffledFrameDataset(cfg, train_ids, seed=0)
val_ds   = ShuffledFrameDataset(cfg, val_ids,   seed=1)
print(f"{n_episodes} episodes — {len(train_ids)} train / {len(val_ids)} val")

In [ ]:
# IterableDataset: no shuffle arg (order comes from the shuffle buffer), workers get
# disjoint episode shards. batch_size is now *frames* per step, not episodes.
BATCH_SIZE = cfg.training.vae.batch_size
num_workers = min(6, multiprocessing.cpu_count() - 1)
_kwargs = dict(
    batch_size=BATCH_SIZE,
    num_workers=num_workers, persistent_workers=num_workers > 0,
    prefetch_factor=4 if num_workers > 0 else None,
    multiprocessing_context='fork' if num_workers > 0 else None,
)
train_loader = DataLoader(train_ds, **_kwargs)
val_loader   = DataLoader(val_ds,   **_kwargs)

In [13]:
def beta(t):
    if cfg.training.vae.linear_beta:
        return t/(cfg.training.vae.nb_epochs*len(train_loader))
    else:
        return 1

In [8]:
x = next(iter(train_loader))

/Users/matheoledevehat/.local/share/uv/python/cpython-3.11.11-macos-aarch64-none/lib/python3.11/multiprocessing/popen_fork.py:66: RuntimeWarning: lancedb fork support is experimental: the internal async runtime has been reset in the forked child, but a small chance of deadlock remains if other state was mid-operation at fork time. The 'forkserver' or 'spawn' multiprocessing start method is likely a safer alternative.
  self.pid = os.fork()
/Users/matheoledevehat/.local/share/uv/python/cpython-3.11.11-macos-aarch64-none/lib/python3.11/multiprocessing/popen_fork.py:66: RuntimeWarning: lancedb fork support is experimental: the internal async runtime has been reset in the forked child, but a small chance of deadlock remains if other state was mid-operation at fork time. The 'forkserver' or 'spawn' multiprocessing start method is likely a safer alternative.
  self.pid = os.fork()
/Users/matheoledevehat/.local/share/uv/python/cpython-3.11.11-macos-aarch64-none/lib/python3.11/multiprocessing/

In [10]:
#vae(x.to(cfg.device))


torch.Size([4004, 32]) torch.Size([4004, 32]) torch.Size([4004, 32])


(tensor([[[[0.4967, 0.4980, 0.4969,  ..., 0.4982, 0.4964, 0.4982],
           [0.4966, 0.4966, 0.4972,  ..., 0.4946, 0.4965, 0.4960],
           [0.4968, 0.4973, 0.4962,  ..., 0.4995, 0.4970, 0.4991],
           ...,
           [0.4990, 0.4979, 0.4991,  ..., 0.4955, 0.4969, 0.4972],
           [0.4978, 0.4967, 0.4982,  ..., 0.4985, 0.4982, 0.4992],
           [0.4983, 0.4969, 0.4982,  ..., 0.4963, 0.4971, 0.4976]],
 
          [[0.5164, 0.5171, 0.5165,  ..., 0.5203, 0.5167, 0.5181],
           [0.5152, 0.5161, 0.5141,  ..., 0.5158, 0.5176, 0.5154],
           [0.5163, 0.5180, 0.5153,  ..., 0.5210, 0.5163, 0.5175],
           ...,
           [0.5165, 0.5172, 0.5170,  ..., 0.5185, 0.5152, 0.5171],
           [0.5158, 0.5161, 0.5154,  ..., 0.5158, 0.5157, 0.5167],
           [0.5156, 0.5173, 0.5160,  ..., 0.5178, 0.5144, 0.5168]],
 
          [[0.5045, 0.5041, 0.5055,  ..., 0.5055, 0.5035, 0.5044],
           [0.5022, 0.5042, 0.5031,  ..., 0.5025, 0.5033, 0.5021],
           [0.5044, 0.50

In [11]:
x.shape

torch.Size([4004, 3, 64, 64])

/Users/matheoledevehat/Code/worldmodels/ha_schmidhuber/.venv/lib/python3.11/site-packages/trackio/utils.py:27: UserWarning: trackio could not flush buffered remote data for run 'vae{beta=sigmoid,epochs=1}': Server error '522 <none>' for url 'https://trackio.flydexo.com/api/bulk_log_system'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/522. It will retry later if possible.
  warnings.warn(message, *args, **kwargs)


In [ ]:
img_size = cfg.dataset.img_size
optimizer = torch.optim.AdamW(vae.parameters(), cfg.training.vae.lr, weight_decay=0)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, cfg.training.vae.nb_epochs)
step = 0

for epoch in tqdm(range(cfg.training.vae.nb_epochs), desc="Epochs"):
    vae.train()
    for x in train_loader:
        x = x.to(cfg.device)  # (B, 3, H, W) float32 in [0,1], frames from ~dozens of episodes
        x_recon, kl = vae(x)
        recon = F.mse_loss(x_recon, x, reduction='sum')/x.shape[0]
        if cfg.training.vae.free_bits:
            loss = recon + torch.clamp(beta(step) * kl, min=32)
        else:
            loss = recon + kl
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        trackio.log({"training-loss": loss.item(), "training-recon": recon.item(), "training-kl": kl.item()})
        step+=1

    vae.eval()
    with torch.no_grad():
        for x in val_loader:
            x = x.to(cfg.device)
            x_recon, kl = vae(x)
            recon = F.mse_loss(x_recon, x, reduction='sum')/x.shape[0]
            # same objective as training (no beta ramp — val measures the final loss)
            loss = recon + (torch.clamp(kl, min=32) if cfg.training.vae.free_bits else kl)
            trackio.log({"val-loss": loss.item(), "val-recon": recon.item(), "val-kl": kl.item()})
    scheduler.step()

In [8]:
#trackio.finish()

In [9]:
torch.save(vae.state_dict(), f'./vae-{cfg.trackio.run_name}.pt') 